# Script 5 — Análise de Cenários Estratégicos com Gemini
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Consome os artefatos dos Scripts 3 e 4 e usa o **Gemini** para gerar análises executivas
estruturadas (JSON tipado via Pydantic) por empresa × cenário.

**Arquitetura:**
- Prompt Chain-of-Thought para raciocínio financeiro estruturado
- Saída tipada via Pydantic + response_schema (JSON garantido)
- Retry com backoff exponencial para rate limits
- Multi-provedor: Gemini (padrão), Claude, OpenAI, Ollama

**Pré-requisitos:** Scripts 3, 3.1 e 4 executados com sucesso.

## Etapa 0 — Configuração e Logging

In [ ]:
import os
import json
import time
import warnings
import logging
import pickle
from pathlib import Path
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Pastas
PASTA_SAIDA = Path("outputs")
PASTA_CENARIOS = PASTA_SAIDA / "cenarios"
PASTA_LOGS = PASTA_SAIDA / "logs"
PASTA_MODELOS = PASTA_SAIDA / "modelos"

for p in (PASTA_SAIDA, PASTA_CENARIOS, PASTA_LOGS, PASTA_MODELOS):
    p.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)
log_file = PASTA_LOGS / "pipeline_cenarios.log"
fh = logging.FileHandler(log_file, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
logging.getLogger().addHandler(fh)
logger = logging.getLogger("cenarios")

# LLM — tudo explícito aqui, sem depender de arquivo externo
LLM_PROVEDOR = "gemini" 

GEMINI_API_KEY = "COLE_SUA_CHAVE_GEMINI_AQUI"
GEMINI_MODEL = "gemini-3.5-flash"

MAX_RETRIES_LLM = 3
DELAY_ENTRE_EMPS = 2.0
MAX_TOKENS_LLM = 1500
TEMPERATURA_LLM = 0.3

if LLM_PROVEDOR == "gemini" and GEMINI_API_KEY.startswith("COLE_"):
    raise ValueError("Preencha GEMINI_API_KEY na célula de configuração.")

# Horizontes e transformações
_HORIZONTES = ["_ITR_T1", "_ITR_T2", "_ITR_T3", "_DFP"]
_LOG_BASES = {"DRE_3.01", "EBITDA", "BPA_1", "BPA_1.01", "BPP_2.01", "BPP_2.03", "BPP_2"}
_ARC_BASES = {"DFC_MI_6.01", "DRE_3.11"}

LOG_TARGETS = {f"TARGET_{b}{h}" for b in _LOG_BASES for h in _HORIZONTES}
ARC_TARGETS = {f"TARGET_{b}{h}" for b in _ARC_BASES for h in _HORIZONTES}

NOME_VARIAVEL = {
    "DRE_3.01": "Receita Liquida",
    "DRE_3.11": "Lucro Liquido",
    "EBITDA": "EBITDA",
    "BPA_1": "Ativo Total",
    "BPA_1.01": "Ativo Circulante",
    "BPP_2.01": "Passivo Circulante",
    "BPP_2.03": "PL",
    "BPP_2": "Passivo Total",
    "DFC_MI_6.01": "FCO",
}

def get_transform(target: str) -> str:
    if target in LOG_TARGETS:
        return "log1p"
    if target in ARC_TARGETS:
        return "arcsinh"
    return "none"

def inv_transform(y, t: str = "none"):
    y = np.asarray(y, float)
    if t == "log1p":
        return np.expm1(y)
    if t == "arcsinh":
        return np.sinh(y)
    return y

logger.info("Etapa 0 OK | provedor=%s | modelo=%s", LLM_PROVEDOR, GEMINI_MODEL)
print(f"OK Etapa 0 | provedor={LLM_PROVEDOR} | modelo={GEMINI_MODEL}")

## Etapa 1 — Carregamento de Artefatos

Carrega artefatos dos Scripts 3, 3.1 e 4:
- **Predições prospectivas** (Script 3): valores preditos por empresa × target
- **KPIs derivados** (Script 3.1): margens, ROE, endividamento, Z''-Score
- **Monte Carlo** (Script 4): intervalos P5-P95 por empresa
- **Empresas destaque** (Script 4): empresa representativa por setor

In [ ]:
def _load(nome, req=False):
    p = PASTA_SAIDA / nome
    if not p.exists():
        if req:
            raise FileNotFoundError(f"{nome} nao encontrado. Execute os Scripts 3/4 antes.")
        logger.warning("%s ausente.", nome)
        return None
    try:
        if nome.endswith(".parquet"):
            obj = pd.read_parquet(p)
        elif nome.endswith(".csv"):
            obj = pd.read_csv(p)
        elif nome.endswith(".pkl"):
            with open(p, "rb") as f:
                obj = pickle.load(f)
        else:
            raise ValueError(f"Extensao nao suportada: {nome}")
        n = f"{len(obj):,} linhas" if hasattr(obj, "__len__") else type(obj).__name__
        logger.info("Carregado: %-40s %s", nome, n)
        return obj
    except Exception as e:
        logger.error("Erro ao carregar %s: %s", nome, e)
        return None

dataset = _load("dataset_cvm_consolidado.parquet", req=True)
melhores = _load("melhores_modelos.pkl", req=True)
if melhores is None:
    melhores = {}

sel_features = _load("selected_features_por_target.pkl", req=True)
if sel_features is None:
    sel_features = {}

df_prosp = _load("predicoes_prospectivas.parquet")
df_zscore = _load("altman_zscore.parquet")
df_stress = _load("analise_estresse_mc.csv")

emps_destaque = _load("empresas_destaque.pkl")
if emps_destaque is None:
    emps_destaque = {}

# Fallback KPIs derivados em paths alternativos
df_kpis_der = _load("painel_avaliacao/b31_kpis_derivados_2026.csv")
if df_kpis_der is None:
    df_kpis_der = _load("kpis_derivados_2026.csv")

# Mapas de lookup
_b = dataset.drop_duplicates("CNPJ_CIA") if "CNPJ_CIA" in dataset.columns else pd.DataFrame()
mapa_nome = _b.set_index("CNPJ_CIA")["NOME_CIA"].to_dict() if "NOME_CIA" in _b.columns else {}
mapa_setor = _b.set_index("CNPJ_CIA")["SETOR"].to_dict() if "SETOR" in _b.columns else {}
mapa_cnpj = {v: k for k, v in mapa_nome.items()}

# Empresas de análise: usa destaque do Script 4; fallback automático
EMPRESAS_ANALISE = {}

if emps_destaque:
    for setor, info in emps_destaque.items():
        nome = info["nome"] if isinstance(info, dict) else info
        cnpj = info.get("cnpj") if isinstance(info, dict) else mapa_cnpj.get(nome)
        if nome and cnpj:
            EMPRESAS_ANALISE[nome] = {"cnpj": cnpj, "setor": setor}
    print(f"Empresas carregadas do Script 4: {len(EMPRESAS_ANALISE)}")
else:
    logger.warning("empresas_destaque.pkl ausente — selecionando por cobertura")
    if "CNPJ_CIA" in dataset.columns and "SETOR" in dataset.columns:
        for setor, grp in dataset.groupby("SETOR"):
            cobertura = grp.groupby("CNPJ_CIA").size()
            cnpj_melhor = cobertura.idxmax()
            nome_melhor = mapa_nome.get(cnpj_melhor, cnpj_melhor)
            EMPRESAS_ANALISE[nome_melhor] = {"cnpj": cnpj_melhor, "setor": setor}

TARGETS_DFP = sorted(set(melhores.keys()) & {f"TARGET_{b}_DFP" for b in NOME_VARIAVEL})

print()
print("=== Empresas para analise de cenarios ===")
for nome, info in EMPRESAS_ANALISE.items():
    print(f"  {nome:<35} setor={info['setor']}")
print(f"\nTargets DFP disponiveis: {len(TARGETS_DFP)}")

## Etapa 2 — Definição dos Cenários Estratégicos

Quatro cenários com diferentes graus de intensidade.
Os ajustes são aplicados por correspondência de substring no nome da feature:
ex. `'endividamento'` afeta qualquer feature cujo nome contenha essa palavra.

In [ ]:
CENARIOS = {
    'Expansao_Alavancagem': {
        'objetivo': 'Avaliar o impacto de uma nova estrutura de capital com aumento de dívida sobre liquidez, cobertura de juros e flexibilidade financeira.',
        'mecanismo_economico': (
            'A empresa capta dívida de longo prazo equivalente a 0,5x o PL. '
            'O choque eleva alavancagem, pressiona encargos financeiros e reduz a folga de liquidez no curto prazo, '
            'com possível contrapartida de expansão de capacidade no médio prazo.'
        ),
        'descricao': (
            'Captação de nova dívida LP equivalente a 0,5x o PL. '
            'Aumenta endividamento e pressiona cobertura de juros e liquidez.'
        ),
        'intensidade': 'moderado',
        'prioridade_analitica': [
            'liquidez',
            'alavancagem',
            'cobertura de juros',
            'solvência',
            'capacidade de investimento',
        ],
        'hipoteses': [
            'Emissão de debêntures ou CRI/CRA no mercado doméstico',
            'Taxa de captação: CDI + 1,5% a.a.',
            'Prazo: 5 anos com carência de 2 anos',
            'Uso dos recursos: expansão de capacidade produtiva',
        ],
        'restricoes': [
            'Não assumir melhora operacional automática.',
            'Não assumir refinanciamento sem evidência.',
            'Distinguir efeito de curto prazo do benefício potencial de médio prazo.',
        ],
        'ajustes': {
            'endividamento': lambda x: x * 1.30,
            'alavancagem_de': lambda x: x * 1.50,
            'liquidez_corrente': lambda x: x * 0.90,
            'liquidez_imediata': lambda x: x * 0.85,
            'cobertura_juros': lambda x: x * 0.75,
            'divida_lp': lambda x: x * 1.50,
        },
    },

    'Eficiencia_Operacional': {
        'objetivo': 'Simular ganhos de eficiência por redução de custos e avaliar efeito sobre margens e retorno.',
        'mecanismo_economico': (
            'Redução de custos operacionais e despesas administrativas, elevando margens operacionais e líquidas '
            'sem mudança relevante de receita no curto prazo.'
        ),
        'descricao': (
            'Redução de 10% nos custos operacionais via automação e renegociação de contratos. '
            'Melhora margens sem alterar receita.'
        ),
        'intensidade': 'conservador',
        'prioridade_analitica': [
            'margem bruta',
            'margem EBITDA',
            'margem líquida',
            'ROE',
            'ROA',
        ],
        'hipoteses': [
            'Redução de 10% no CPV',
            'Redução de 8% nas despesas administrativas',
            'Sem impacto na receita no curto prazo',
            'Implementação em 12 meses',
        ],
        'restricoes': [
            'Não extrapolar eficiência acima de limites plausíveis.',
            'Não assumir ganho estrutural permanente sem evidência.',
        ],
        'ajustes': {
            'margem_ebit': lambda x: min(x * 1.15, 0.60) if x >= 0 else x * 0.85,
            'margem_ebitda': lambda x: min(x * 1.12, 0.70) if x >= 0 else x * 0.88,
            'margem_liquida': lambda x: min(x * 1.10, 0.50) if x >= 0 else x * 0.90,
            'roe': lambda x: x * 1.08,
            'roa': lambda x: x * 1.06,
            'giro_ativo': lambda x: x * 1.02,
        },
    },

    'CAPEX_Capacidade': {
        'objetivo': 'Avaliar o efeito de aumento de CAPEX sobre liquidez de curto prazo, geração de caixa e expansão futura.',
        'mecanismo_economico': (
            'A empresa intensifica investimentos em capacidade, consumindo caixa no curto prazo e pressionando '
            'indicadores de liquidez e geração operacional, com benefício potencial no médio prazo.'
        ),
        'descricao': (
            'Aumento de 30% no CAPEX. '
            'Pressiona liquidez de curto prazo e FCO, mas expande capacidade futura.'
        ),
        'intensidade': 'moderado',
        'prioridade_analitica': [
            'liquidez',
            'FCO',
            'endividamento',
            'ativo total',
            'giro do ativo',
        ],
        'hipoteses': [
            'CAPEX adicional 50% caixa próprio + 50% financiamento',
            'Depreciação adicional a partir do 2º ano',
            'Ganho de capacidade produtiva de 20% em 3 anos',
            'Sem retorno de receita no ano 1',
        ],
        'restricoes': [
            'Não assumir retorno imediato do investimento.',
            'Separar pressão de caixa de eventual ganho de capacidade.',
        ],
        'ajustes': {
            'giro_ativo': lambda x: x * 0.92,
            'liquidez_corrente': lambda x: x * 0.85,
            'liquidez_imediata': lambda x: x * 0.80,
            'fco_receita': lambda x: x * 0.90,
            'capex_receita': lambda x: x * 1.30,
            'ativo_total': lambda x: x * 1.08,
        },
    },

    'Choque_Macro_Adverso': {
        'objetivo': 'Estressar a empresa em um ambiente macro adverso e medir fragilidade em liquidez, margem e risco de solvência.',
        'mecanismo_economico': (
            'O choque macro reduz receita, comprime margens, eleva custo da dívida e deteriora cobertura de juros '
            'e liquidez, com efeito mais severo em empresas já alavancadas.'
        ),
        'descricao': (
            'Estresse macroeconômico: SELIC a 15%, dólar a R$6,50, PIB crescendo 0,5%. '
            'Pressão em toda a cadeia financeira.'
        ),
        'intensidade': 'severo',
        'prioridade_analitica': [
            'receita',
            'margem líquida',
            'margem EBITDA',
            'cobertura de juros',
            'liquidez',
            'endividamento',
        ],
        'hipoteses': [
            'SELIC: 15% a.a. (alta de ~300bps)',
            'Dólar: R$ 6,50 (+15% vs base)',
            'PIB: +0,5% (desaceleração forte)',
            'IPCA: 8% a.a.',
            'Queda de 8% na receita por redução de consumo',
        ],
        'restricoes': [
            'Não supor repasse integral de custos.',
            'Não supor preservação de margem sem evidência.',
            'Não tratar o choque como temporário sem justificativa.',
        ],
        'ajustes': {
            'receita': lambda x: x * 0.92,
            'margem_liquida': lambda x: x * 0.75 if x >= 0 else x * 1.25,
            'margem_ebitda': lambda x: x * 0.85 if x >= 0 else x * 1.15,
            'cobertura_juros': lambda x: x * 0.65,
            'liquidez_corrente': lambda x: x * 0.80,
            'endividamento': lambda x: x * 1.20,
            'roe': lambda x: x * 0.70 if x >= 0 else x * 1.30,
        },
    },
}

##  Etapa 3 — Funções de Simulação

Para cada empresa × cenário:
1. Busca o último vetor de features no dataset histórico
2. Aplica os ajustes do cenário às features correspondentes
3. Re-prediz cada target DFP com o modelo treinado (Script 3)
4. Calcula o delta percentual vs. predição base (sem ajuste)

In [ ]:
def ultima_linha_empresa(cnpj):
    df_e = dataset[dataset["CNPJ_CIA"] == cnpj]
    if df_e.empty:
        return None, None
    if "ANO" in df_e.columns:
        df_e = df_e.sort_values("ANO")
    ul = df_e.iloc[-1]
    ano = int(ul["ANO"]) if "ANO" in ul.index else None
    return ul, ano


def aplicar_cenario_ao_vetor(row_base, cenario: dict, feats: list) -> np.ndarray:
    vals = []
    for f in feats:
        v = row_base.get(f)
        v = float(v) if pd.notna(v) else 0.0
        for padrao, func in cenario["ajustes"].items():
            if padrao.lower() in f.lower():
                try:
                    v = func(v)
                except Exception:
                    pass
                break
        vals.append(v)
    return np.array(vals, dtype=float)


def prever_target(target: str, x_sub: np.ndarray, feats_sub: list) -> Optional[float]:
    alg = melhores.get(target)
    if not alg:
        return None

    cam = PASTA_MODELOS / f"modelo_{target}_{alg}.pkl"
    if not cam.exists():
        return None

    try:
        obj = joblib.load(cam)
        modelo = obj["modelo"] if isinstance(obj, dict) and "modelo" in obj else obj

        n_exp = None
        for step in (getattr(modelo, "steps", None) or []):
            est = step[1] if isinstance(step, tuple) else step
            if hasattr(est, "n_features_in_"):
                n_exp = est.n_features_in_
                break
        if n_exp is None and hasattr(modelo, "n_features_in_"):
            n_exp = modelo.n_features_in_

        x = np.asarray(x_sub, dtype=float).copy()
        if n_exp is not None:
            if len(x) < n_exp:
                x = np.pad(x, (0, n_exp - len(x)), constant_values=0.0)
            elif len(x) > n_exp:
                x = x[:n_exp]

        x = np.nan_to_num(x.reshape(1, -1), nan=0.0)
        y_raw = float(modelo.predict(x)[0])
        return float(inv_transform([y_raw], get_transform(target))[0])

    except Exception as e:
        logger.debug("prever_target | target=%s | %s", target, e)
        return None


def simular_empresa_cenario(
    nome_emp: str,
    info_emp: dict,
    cenario_nome: str,
    cenario: dict
) -> dict:
    cnpj = info_emp["cnpj"]
    setor = info_emp["setor"]

    row_base, ano_base = ultima_linha_empresa(cnpj)
    if row_base is None:
        return {}

    resultados = {}

    # preserva a ordem das features do target, sem ordenar globalmente
    for target in TARGETS_DFP:
        feats_t = [f for f in sel_features.get(target, []) if f in dataset.columns]
        if not feats_t:
            continue

        x_base = np.array(
            [float(row_base.get(f, np.nan)) if pd.notna(row_base.get(f)) else 0.0 for f in feats_t],
            dtype=float,
        )
        x_cen = aplicar_cenario_ao_vetor(row_base, cenario, feats_t)

        y_base = prever_target(target, x_base, feats_t)
        y_cen = prever_target(target, x_cen, feats_t)

        if y_base is None or y_cen is None:
            continue
        if not (np.isfinite(y_base) and np.isfinite(y_cen)):
            continue

        delta = (y_cen - y_base) / abs(y_base) * 100 if abs(y_base) > 1e-9 else 0.0
        base_str = target.replace("TARGET_", "").replace("_DFP", "")

        resultados[target] = {
            "variavel": NOME_VARIAVEL.get(base_str, base_str),
            "y_base_bi": round(y_base / 1e6, 3),
            "y_cen_bi": round(y_cen / 1e6, 3),
            "delta_pct": round(delta, 2),
            "algoritmo": melhores.get(target, "?"),
        }

    zscore_info = {}
    if df_zscore is not None and "CNPJ_CIA" in df_zscore.columns:
        dz = df_zscore[df_zscore["CNPJ_CIA"] == cnpj].dropna(subset=["altman_z_pp"])
        if not dz.empty and "ANO" in dz.columns:
            last = dz.sort_values("ANO").iloc[-1]
            zscore_info = {
                "z_pp_ultimo": round(float(last["altman_z_pp"]), 3),
                "zona_atual": str(last.get("zona_altman", "N/D")),
            }

    mc_info = {}
    if df_stress is not None and "cnpj" in df_stress.columns:
        dm = df_stress[df_stress["cnpj"] == cnpj]
        if not dm.empty:
            mc_info["cv_medio"] = round(float(dm["cv"].mean()), 3)
            if "target" in dm.columns and "p_negativo" in dm.columns:
                p_neg = dm[dm["target"].str.contains("DRE_3.11", na=False)]["p_negativo"]
                if not p_neg.empty:
                    mc_info["p_negativo_lucro"] = round(float(p_neg.mean()), 3)

    return {
        "empresa": nome_emp,
        "cnpj": cnpj,
        "setor": setor,
        "ano_base": ano_base,
        "cenario": cenario_nome,
        "intensidade": cenario["intensidade"],
        "resultados_financeiros": resultados,
        "zscore": zscore_info,
        "monte_carlo": mc_info,
    }

print("OK Funcoes de simulacao prontas")

##  Etapa 4 — Módulo LLM com Saída Estruturada

Usa a nova API `google-genai` com:
- `response_schema` Pydantic: JSON garantido sem parsing frágil
- Chain-of-Thought no prompt: raciocínio antes da conclusão
- Retry com backoff exponencial
- Provedor: Gemini

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ImpactoFinanceiro(BaseModel):
    variavel: str = Field(description='Nome da variavel financeira')
    variacao_pct: float = Field(description='Variacao percentual projetada')
    interpretacao: str = Field(description='Interpretacao executiva em 1 frase')

class AnaliseEstrategica(BaseModel):
    empresa: str = Field(description='Nome da empresa')
    cenario: str = Field(description='Nome do cenario')
    resumo_executivo: str = Field(description='Sintese do impacto em 2-3 frases')
    principais_impactos: List[ImpactoFinanceiro] = Field(description='Ate 4 variaveis com maior impacto')
    riscos: List[str] = Field(description='Lista de 2-3 riscos especificos')
    oportunidades: List[str] = Field(description='Lista de 1-2 oportunidades')
    recomendacao: str = Field(description='Recomendacao estrategica (max 2 frases)')
    score_impacto: int = Field(description='Score 1=leve a 10=critico', ge=1, le=10)
    zona_risco_projetada: str = Field(description='Segura|Cinza|Insolvencia|Indeterminado')


def construir_prompt(sim: dict) -> str:
    emp = sim['empresa']
    setor = sim['setor']
    ano = sim['ano_base']
    cen = sim['cenario']
    cfg = CENARIOS.get(cen, {})

    objetivo = cfg.get('objetivo', 'N/A')
    mecanismo = cfg.get('mecanismo_economico', cfg.get('descricao', cen))
    intensidade = cfg.get('intensidade', 'N/A')
    prioridade = cfg.get('prioridade_analitica', [])
    hip = cfg.get('hipoteses', [])
    restricoes = cfg.get('restricoes', [])

    impacto_prioritario = []
    for tgt, d in sorted(sim['resultados_financeiros'].items(), key=lambda x: abs(x[1]['delta_pct']), reverse=True):
        sinal = '+' if d['delta_pct'] >= 0 else ''
        impacto_prioritario.append(
            f"- {d['variavel']}: base={d['y_base_bi']:.2f} Bi | cenário={d['y_cen_bi']:.2f} Bi | delta={sinal}{d['delta_pct']:.1f}%"
        )
    impactos_txt = '\n'.join(impacto_prioritario) or '- Sem dados de predicao'

    z = sim.get('zscore', {})
    mc = sim.get('monte_carlo', {})

    z_txt = (
        f"Z'' atual={z.get('z_pp_ultimo', 'N/D')} (Zona: {z.get('zona_atual', 'N/D')})"
        if z else "Z''-Score: nao disponivel"
    )
    mc_txt = (
        f"CV medio={mc.get('cv_medio', 'N/D')}"
        + (f" | P(Lucro<0)={mc['p_negativo_lucro']:.1%}" if mc.get('p_negativo_lucro') is not None else '')
        if mc else 'Monte Carlo: nao disponivel'
    )

    return f"""
Você é um analista financeiro sênior especializado em risco corporativo, valuation e stress testing.

Tarefa:
1. Interpretar o cenário como um choque econômico-financeiro.
2. Explicar o mecanismo causal do choque sobre a empresa.
3. Comparar claramente base vs cenário.
4. Priorizar liquidez, alavancagem, margens, cobertura de juros e risco de insolvência.
5. Não inventar dados. Use apenas os números fornecidos.
6. Se uma conclusão não puder ser sustentada, diga explicitamente que é uma inferência.
7. Responder em JSON válido conforme o schema solicitado.

Contexto da empresa:
- Empresa: {emp}
- Setor: {setor}
- Ano-base: {ano}

Definição do cenário:
- Nome: {cen}
- Intensidade: {intensidade}
- Objetivo: {objetivo}
- Mecanismo econômico: {mecanismo}

Prioridades analíticas:
{chr(10).join(f"- {p}" for p in prioridade) if prioridade else "- N/A"}

Hipóteses do cenário:
{chr(10).join(f"- {h}" for h in hip) if hip else "- N/A"}

Restrições de interpretação:
{chr(10).join(f"- {r}" for r in restricoes) if restricoes else "- N/A"}

Evidências numéricas:
{impactos_txt}

Risco atual:
- {z_txt}
- {mc_txt}

Saída esperada:
- resumo executivo objetivo
- principais impactos com interpretação causal
- riscos e oportunidades
- recomendação executiva
- score de impacto de 1 a 10
- zona de risco projetada
""".strip()


def _gemini_structured(prompt):
    try:
        from google import genai as gai
        from google.genai import types as gt

        client = gai.Client(api_key=GEMINI_API_KEY)
        resp = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=gt.GenerateContentConfig(
                response_mime_type='application/json',
                response_schema=AnaliseEstrategica,
                temperature=TEMPERATURA_LLM,
                max_output_tokens=MAX_TOKENS_LLM,
            ),
        )

        if getattr(resp, 'parsed', None):
            return resp.parsed

        if getattr(resp, 'text', None):
            return AnaliseEstrategica(**json.loads(resp.text))

        return None

    except Exception as e:
        logger.warning('Gemini structured falhou: %s', e)
        return None


def _gemini_texto(prompt):
    schema_hint = (
        '\nResponda EXCLUSIVAMENTE com JSON valido:\n'
        '{"empresa":"","cenario":"","resumo_executivo":"","principais_impactos":'
        '[{"variavel":"","variacao_pct":0,"interpretacao":""}],"riscos":[],'
        '"oportunidades":[],"recomendacao":"","score_impacto":5,'
        '"zona_risco_projetada":"Indeterminado"}'
    )
    try:
        from google import genai as gai
        from google.genai import types as gt

        client = gai.Client(api_key=GEMINI_API_KEY)
        resp = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt + schema_hint,
            config=gt.GenerateContentConfig(
                temperature=TEMPERATURA_LLM,
                max_output_tokens=MAX_TOKENS_LLM,
            ),
        )

        raw = (resp.text or '').strip()
        if raw.startswith('```'):
            raw = '\n'.join(raw.split('\n')[1:])
            if raw.endswith('```'):
                raw = raw[:-3].strip()

        return AnaliseEstrategica(**json.loads(raw))

    except Exception as e:
        logger.error('Gemini texto falhou: %s', e)
        return None



def chamar_llm(sim: dict):
    prompt = construir_prompt(sim)
    for tentativa in range(1, MAX_RETRIES_LLM + 1):
        try:
            if LLM_PROVEDOR == 'gemini':
                result = _gemini_structured(prompt) or _gemini_texto(prompt)
            else:
                return None, prompt

            if result is not None:
                return result, prompt

        except Exception as e:
            wait = 2 ** tentativa
            logger.warning(
                'Tentativa %d/%d | %s | aguardando %ds',
                tentativa, MAX_RETRIES_LLM, e, wait
            )
            time.sleep(wait)

    logger.error('Todas tentativas falharam | %s | %s', sim.get('empresa'), sim.get('cenario'))
    return None, prompt

print(
    'OK Modulo LLM | schema=AnaliseEstrategica |',
    f'campos={len(AnaliseEstrategica.model_fields)} | retries={MAX_RETRIES_LLM}'
)

##  Etapa 5 — Execução: Simulação + Análise LLM

Loop principal: empresa × cenário → simulação numérica → chamada LLM → coleta.

In [ ]:
todos_resultados = []
todos_prompts = {}
n_total = len(EMPRESAS_ANALISE) * len(CENARIOS)
n_proc = 0

print(f"Iniciando: {len(EMPRESAS_ANALISE)} empresas x {len(CENARIOS)} cenarios = {n_total} analises")
print("=" * 70)

for nome_emp, info_emp in EMPRESAS_ANALISE.items():
    print(f"\nEMPRESA: {nome_emp} | SETOR: {info_emp['setor']}")
    print("-" * 50)

    for cenario_nome, cenario in CENARIOS.items():
        n_proc += 1
        print(f"  [{n_proc:>3}/{n_total}] {cenario_nome} [{cenario['intensidade']}]", end=" ")

        sim = simular_empresa_cenario(nome_emp, info_emp, cenario_nome, cenario)
        if not sim or not sim.get("resultados_financeiros"):
            print("-> SKIP (sem dados de predicao)")
            continue

        n_v = len(sim["resultados_financeiros"])
        print(f"-> {n_v} vars", end=" ")

        analise, prompt = chamar_llm(sim)
        todos_prompts[f"{nome_emp}__{cenario_nome}"] = prompt

        rec = {
            "empresa": nome_emp,
            "cnpj": info_emp["cnpj"],
            "setor": info_emp["setor"],
            "ano_base": sim.get("ano_base"),
            "cenario": cenario_nome,
            "intensidade": cenario["intensidade"],
            "llm_ok": analise is not None,
            "score_impacto": getattr(analise, "score_impacto", None),
            "zona_risco_projetada": getattr(analise, "zona_risco_projetada", None),
            "resumo_executivo": getattr(analise, "resumo_executivo", None),
            "recomendacao": getattr(analise, "recomendacao", None),
            "riscos": " | ".join(getattr(analise, "riscos", [])),
            "oportunidades": " | ".join(getattr(analise, "oportunidades", [])),
            "z_pp_ultimo": sim.get("zscore", {}).get("z_pp_ultimo"),
            "zona_atual": sim.get("zscore", {}).get("zona_atual"),
            "cv_medio_mc": sim.get("monte_carlo", {}).get("cv_medio"),
            "p_negativo_lucro_mc": sim.get("monte_carlo", {}).get("p_negativo_lucro"),
        }

        for tgt, d in sim["resultados_financeiros"].items():
            col = d["variavel"].replace(" ", "_").replace("/", "_")
            rec[f"{col}_base_bi"] = d["y_base_bi"]
            rec[f"{col}_cen_bi"] = d["y_cen_bi"]
            rec[f"{col}_delta_pct"] = d["delta_pct"]

        if analise:
            for i, imp in enumerate(getattr(analise, "principais_impactos", [])[:4], 1):
                rec[f"impacto_{i}_variavel"] = imp.variavel
                rec[f"impacto_{i}_delta_pct"] = imp.variacao_pct
                rec[f"impacto_{i}_interpretacao"] = imp.interpretacao

        todos_resultados.append(rec)
        status = (
            f"score={analise.score_impacto}/10 zona={analise.zona_risco_projetada} OK"
            if analise else "LLM FALHOU"
        )
        print(f"-> {status}")
        time.sleep(DELAY_ENTRE_EMPS)

print()
n_ok = sum(1 for r in todos_resultados if r.get("llm_ok"))
print(f"OK Execucao concluida | total={len(todos_resultados)} | llm_ok={n_ok}")

##  Etapa 6 — Persistência dos Resultados

Salva em CSV, Excel (com abas por cenário), JSON completo e PKL para o Script 5.1.

In [ ]:
if todos_resultados:
    df_res = pd.DataFrame(todos_resultados)

    csv_path = PASTA_CENARIOS / "resultados_cenarios.csv"
    df_res.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"OK CSV  : {csv_path} ({len(df_res)} linhas x {len(df_res.columns)} cols)")

    xlsx_path = PASTA_CENARIOS / "resultados_cenarios.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_res.to_excel(writer, sheet_name="Resultados", index=False)

        cols_res = [
            "empresa",
            "setor",
            "cenario",
            "intensidade",
            "score_impacto",
            "zona_risco_projetada",
            "resumo_executivo",
            "recomendacao",
        ]
        df_res[[c for c in cols_res if c in df_res.columns]].to_excel(
            writer,
            sheet_name="Resumo_Executivo",
            index=False,
        )

        if "score_impacto" in df_res.columns:
            (df_res[df_res["llm_ok"] == True]
             .sort_values("score_impacto", ascending=False)
             [["empresa", "setor", "cenario", "score_impacto", "zona_risco_projetada", "zona_atual"]]
             .to_excel(writer, sheet_name="Ranking_Impacto", index=False))

        for cen_nome in CENARIOS:
            dc = df_res[df_res["cenario"] == cen_nome]
            if not dc.empty:
                dc.to_excel(writer, sheet_name=cen_nome[:31], index=False)

    print(f"OK XLSX : {xlsx_path}")

    json_out = {
        f'{r["empresa"]}__{r["cenario"]}': {
            "empresa": r["empresa"],
            "setor": r["setor"],
            "cenario": r["cenario"],
            "ano_base": r.get("ano_base"),
            "llm_ok": r.get("llm_ok"),
            "score": r.get("score_impacto"),
            "zona_proj": r.get("zona_risco_projetada"),
            "resumo": r.get("resumo_executivo"),
            "recomendacao": r.get("recomendacao"),
            "riscos": (r.get("riscos", "").split(" | ") if r.get("riscos") else []),
            "oportunidades": (r.get("oportunidades", "").split(" | ") if r.get("oportunidades") else []),
            "prompt": todos_prompts.get(f'{r["empresa"]}__{r["cenario"]}', ""),
        }
        for r in todos_resultados
    }

    json_path = PASTA_CENARIOS / "analises_llm.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_out, f, ensure_ascii=False, indent=2)
    print(f"OK JSON : {json_path}")

    pkl_path = PASTA_SAIDA / "resultados_cenarios.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump({"df": df_res, "raw": todos_resultados, "prompts": todos_prompts}, f)
    print(f"OK PKL  : {pkl_path}")
else:
    df_res = pd.DataFrame()
    print("AVISO: nenhum resultado — verifique os logs.")

##  Etapa 7 — Tabelas-Resumo Comparativas

In [ ]:
if not df_res.empty and "score_impacto" in df_res.columns:
    df_ok = df_res[df_res["llm_ok"] == True].copy()

    print("=== RANKING DE IMPACTO ===")
    print(
        df_ok.sort_values("score_impacto", ascending=False)
        [["empresa", "setor", "cenario", "score_impacto", "zona_risco_projetada", "zona_atual"]]
        .to_string(index=False)
    )

    print("\n=== SCORE MEDIO POR EMPRESA ===")
    print(
        df_ok.groupby("empresa")["score_impacto"]
        .agg(["mean", "max", "min"]).round(2)
        .sort_values("mean", ascending=False).to_string()
    )

    print("\n=== SCORE MEDIO POR CENARIO ===")
    print(
        df_ok.groupby(["cenario", "intensidade"])["score_impacto"]
        .agg(["mean", "max"]).round(2)
        .sort_values("mean", ascending=False).to_string()
    )

    if "zona_atual" in df_ok.columns and "zona_risco_projetada" in df_ok.columns:
        df_ok["mudou_zona"] = df_ok["zona_atual"] != df_ok["zona_risco_projetada"]
        mudancas = df_ok[df_ok["mudou_zona"]][
            ["empresa", "cenario", "zona_atual", "zona_risco_projetada", "score_impacto"]
        ]
        print("\n=== MUDANCAS DE ZONA PROJETADAS ===")
        print(
            mudancas.sort_values("score_impacto", ascending=False).to_string(index=False)
            if not mudancas.empty else "  Nenhuma mudanca de zona projetada."
        )

    print("\n=== ANALISES LLM — RESUMO EXECUTIVO ===")
    for _, row in df_ok.sort_values(["empresa", "score_impacto"], ascending=[True, False]).iterrows():
        print(f"\n[{row['empresa']} | {row['cenario']} | score={row['score_impacto']}/10]")
        print(f"  Zona: {row.get('zona_atual', '?')} -> {row.get('zona_risco_projetada', '?')}")
        print(f"  Resumo: {row.get('resumo_executivo', '')}")
        print(f"  Recomendacao: {row.get('recomendacao', '')}")
        for r in (row.get("riscos", "") or "").split(" | ")[:2]:
            if r:
                print(f"  Risco: {r}")

## Etapa 8 — Relatório Final

In [ ]:
from datetime import datetime

print("=" * 70)
print("RELATORIO FINAL — 05_cvm_cenarios.ipynb")
print("=" * 70)
print(f"Executado em : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Provedor LLM : {LLM_PROVEDOR}")
print(f"Empresas     : {len(EMPRESAS_ANALISE)}")
print(f"Cenarios     : {len(CENARIOS)}")

n_tot = len(todos_resultados)
n_ok = sum(1 for r in todos_resultados if r.get("llm_ok"))
print(f"Analises     : {n_tot} | LLM OK: {n_ok}/{n_tot}")
print()

check = [
    ("Dataset carregado", dataset is not None),
    ("Predicoes prospectivas", df_prosp is not None),
    ("Z-Score calculado", df_zscore is not None),
    ("Monte Carlo disponivel", df_stress is not None),
    ("Cenarios simulados", n_tot > 0),
    ("Analises LLM geradas", n_ok > 0),
    ("CSV exportado", (PASTA_CENARIOS / "resultados_cenarios.csv").exists()),
    ("Excel exportado", (PASTA_CENARIOS / "resultados_cenarios.xlsx").exists()),
    ("JSON analises exportado", (PASTA_CENARIOS / "analises_llm.json").exists()),
    ("PKL para Script 5.1", (PASTA_SAIDA / "resultados_cenarios.pkl").exists()),
]

all_ok = True
for desc, status in check:
    ok = bool(status)
    all_ok = all_ok and ok
    print(f"  [{'OK' if ok else 'FALTA'}] {desc}")

rel = {
    "timestamp": datetime.now().isoformat(),
    "provedor": LLM_PROVEDOR,
    "modelo": GEMINI_MODEL,
    "empresas": list(EMPRESAS_ANALISE.keys()),
    "cenarios": list(CENARIOS.keys()),
    "n_analises": n_tot,
    "n_llm_ok": n_ok,
    "checklist": {d: bool(s) for d, s in check},
}

rel_path = PASTA_LOGS / "relatorio_cenarios.json"
with open(rel_path, "w", encoding="utf-8") as f:
    json.dump(rel, f, ensure_ascii=False, indent=2)

print()
print("STATUS:", "CONCLUIDO COM SUCESSO" if all_ok else "CONCLUIDO COM PENDENCIAS")
print(f"Log: {rel_path}")